# Step 1: Setup and Imports

In this project, we build an AI Resume Screening System.

The system will:
- Extract skills from resumes
- Match them with job description
- Assign a score
- Provide explanation

We will use HuggingFace (no API key needed) for simplicity.

In [1]:
!pip install transformers

from transformers import pipeline
from langchain_core.prompts import PromptTemplate

# Step 2: Define Inputs

We define:
- 1 Job Description
- 3 Resumes (Strong, Average, Weak)

These will be used to test our system.

In [2]:
# Job Description
job_description = """
We are hiring a Data Scientist.

Requirements:
- Python, Machine Learning, Deep Learning
- Pandas, NumPy, Scikit-learn
- NLP (optional)
- Data visualization
- 2+ years experience
"""

# Strong Resume
resume_strong = """
Skills: Python, Machine Learning, Deep Learning, NLP, Pandas, NumPy, Scikit-learn
Experience: 3 years Data Scientist
Tools: Python, Jupyter
"""

# Average Resume
resume_average = """
Skills: Python, Pandas, basic Machine Learning
Experience: 1 year Data Analyst
Tools: Excel, Python
"""

# Weak Resume
resume_weak = """
Skills: MS Excel
Experience: Fresher
Tools: None
"""

# Step 3: Load Model

We load a HuggingFace model to generate outputs.

Due to environment compatibility, we use text-generation pipeline.

This model will process prompts and generate results for our system.

In [5]:
from transformers import pipeline

# Correct working pipeline
hf_pipeline = pipeline(
    "text-generation",
    model="gpt2"
)

# Helper function
def run_llm(prompt):
    result = hf_pipeline(
        prompt,
        max_new_tokens=100,
        return_full_text=False
    )
    return result[0]["generated_text"]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
print(run_llm("Explain machine learning in simple terms"))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 and use it to generate data on the world.

It is also good to learn how to build and use R packages as well, which will give you a better understanding of what you are building and how to make it better.

What is R?

R is an open-source language that provides a complete set of tools for building and managing applications. R is very similar to Python in that it provides a good set of tools for building, debugging, and manipulating R code.


# Step 4: Skill Extraction

In this step, we extract important information from the resume.

We extract:
- Skills
- Experience
- Tools

Important:
- Only use information present in resume
- Do NOT assume anything

This step converts raw resume text into structured data.

In [10]:
def extract_info(resume):

    lines = resume.split("\n")

    skills = ""
    experience = ""
    tools = ""

    for line in lines:
        line = line.strip()

        if "Skills:" in line:
            skills = line.replace("Skills:", "").strip()

        elif "Experience:" in line:
            experience = line.replace("Experience:", "").strip()

        elif "Tools:" in line:
            tools = line.replace("Tools:", "").strip()

    return {
        "Skills": skills,
        "Experience": experience,
        "Tools": tools
    }

# Run extraction
extracted = extract_info(resume_strong)

print("Extracted Data:")
print(extracted)

Extracted Data:
{'Skills': 'Python, Machine Learning, Deep Learning, NLP, Pandas, NumPy, Scikit-learn', 'Experience': '3 years Data Scientist', 'Tools': 'Python, Jupyter'}


In [9]:
def run_llm(prompt):
    result = hf_pipeline(
        prompt,
        max_new_tokens=80,
        return_full_text=False,
        pad_token_id=50256
    )
    return result[0]["generated_text"]

# Step 5: Matching Logic

In this step, we compare the extracted resume data with the job description.

We identify:
- Matching Skills (present in both)
- Missing Skills (required but not present)

This helps evaluate how well the candidate fits the job.

We use rule-based matching to ensure accuracy and avoid hallucination.

In [11]:
def match_skills(extracted, job_description):

    # Convert to lowercase for comparison
    resume_skills = extracted["Skills"].lower().split(", ")
    job_text = job_description.lower()

    matching = []
    missing = []

    for skill in resume_skills:
        if skill in job_text:
            matching.append(skill)

    # Expected skills from job description
    expected_skills = [
        "python", "machine learning", "deep learning",
        "pandas", "numpy", "scikit-learn", "nlp"
    ]

    for skill in expected_skills:
        if skill not in resume_skills:
            missing.append(skill)

    return {
        "Matching Skills": matching,
        "Missing Skills": missing
    }

# Run matching
match_result = match_skills(extracted, job_description)

print("Matching Result:")
print(match_result)

Matching Result:
{'Matching Skills': ['python', 'machine learning', 'deep learning', 'nlp', 'pandas', 'numpy', 'scikit-learn'], 'Missing Skills': []}


# Step 6: Scoring System

In this step, we assign a score (0–100) to the candidate.

Scoring is based on:
- Number of matching skills
- Number of missing skills

Logic:
- More matching → higher score
- More missing → lower score

This converts the matching result into a numerical evaluation.

In [12]:
def calculate_score(match_result):

    matching = len(match_result["Matching Skills"])
    missing = len(match_result["Missing Skills"])

    total = matching + missing

    if total == 0:
        return 0

    score = (matching / total) * 100

    return round(score, 2)


# Run scoring
score = calculate_score(match_result)

print("Score:", score)

Score: 100.0


# Step 7: Explanation

In this step, we explain why a particular score was assigned.

The explanation is based on:
- Matching skills
- Missing skills

This makes the system transparent and helps recruiters understand the decision.

Explainability is an important part of real-world AI systems.

In [13]:
def generate_explanation(match_result, score):

    matching = match_result["Matching Skills"]
    missing = match_result["Missing Skills"]

    explanation = f"""
The candidate has {len(matching)} matching skills: {', '.join(matching)}.

The candidate is missing {len(missing)} important skills: {', '.join(missing)}.

Therefore, the overall score is {score}.
"""

    return explanation


# Run explanation
explanation = generate_explanation(match_result, score)

print("Explanation:")
print(explanation)

Explanation:

The candidate has 7 matching skills: python, machine learning, deep learning, nlp, pandas, numpy, scikit-learn.

The candidate is missing 0 important skills: .

Therefore, the overall score is 100.0.



# Step 8: Testing the Complete Pipeline

In this step, we test the system on:
- Strong candidate
- Average candidate
- Weak candidate

Pipeline:
Resume → Extraction → Matching → Scoring → Explanation

This helps verify that:
- Strong candidate gets high score
- Average gets medium score
- Weak gets low score

In [14]:
resumes = {
    "Strong": resume_strong,
    "Average": resume_average,
    "Weak": resume_weak
}

for name, resume in resumes.items():

    print("\n==============================")
    print("Candidate Type:", name)

    # Step 1: Extract
    extracted = extract_info(resume)

    # Step 2: Match
    match_result = match_skills(extracted, job_description)

    # Step 3: Score
    score = calculate_score(match_result)

    # Step 4: Explanation
    explanation = generate_explanation(match_result, score)

    print("Score:", score)
    print("Explanation:", explanation)


Candidate Type: Strong
Score: 100.0
Explanation: 
The candidate has 7 matching skills: python, machine learning, deep learning, nlp, pandas, numpy, scikit-learn.

The candidate is missing 0 important skills: .

Therefore, the overall score is 100.0.


Candidate Type: Average
Score: 28.57
Explanation: 
The candidate has 2 matching skills: python, pandas.

The candidate is missing 5 important skills: machine learning, deep learning, numpy, scikit-learn, nlp.

Therefore, the overall score is 28.57.


Candidate Type: Weak
Score: 0.0
Explanation: 
The candidate has 0 matching skills: .

The candidate is missing 7 important skills: python, machine learning, deep learning, pandas, numpy, scikit-learn, nlp.

Therefore, the overall score is 0.0.



# Step 9: LangSmith Tracing

In this step, we enable LangSmith tracing to monitor and debug the pipeline.

Tracing helps:
- Track each step of execution
- Debug errors
- Visualize pipeline flow

This is important for production-level AI systems.

In [15]:
!pip install langsmith

In [16]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Resume-Screening-System"

# Paste your LangSmith API key here
os.environ["LANGCHAIN_API_KEY"] = "your_langsmith_api_key"

In [17]:
from langsmith import traceable

@traceable
def run_pipeline(resume):

    extracted = extract_info(resume)
    match_result = match_skills(extracted, job_description)
    score = calculate_score(match_result)
    explanation = generate_explanation(match_result, score)

    return {
        "score": score,
        "explanation": explanation
    }

# Run tracing
run_pipeline(resume_strong)
run_pipeline(resume_average)
run_pipeline(resume_weak)

{'score': 0.0,
 'explanation': '\nThe candidate has 0 matching skills: .\n\nThe candidate is missing 7 important skills: python, machine learning, deep learning, pandas, numpy, scikit-learn, nlp.\n\nTherefore, the overall score is 0.0.\n'}